In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt, os
from PIL import Image
from PIL import ImageDraw
import math, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG, EPOCHS, BS, LR = 128, 100, 64, 2e-4

In [ ]:
class CEDataset(Dataset):
    def __init__(self, root):
        self.files = [os.path.join(root,f) for f in sorted(os.listdir(root))]
        self.tfm = T.Compose([T.Resize((IMG,IMG)), T.ToTensor(), T.Normalize([.5]*3,[.5]*3)])
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = self.tfm(Image.open(self.files[i]).convert('RGB'))
        mask_img = Image.new('L', (IMG,IMG), 0)
        draw = ImageDraw.Draw(mask_img)
        for _ in range(random.randint(1,3)):
            cx,cy,r = random.randint(0,IMG), random.randint(0,IMG), random.randint(10,40)
            nv = random.randint(4,8)
            pts = [(cx+r*math.cos(2*math.pi*k/nv), cy+r*math.sin(2*math.pi*k/nv)) for k in range(nv)]
            draw.polygon(pts, fill=255)
        mask = T.ToTensor()(mask_img).repeat(3,1,1)
        return img*(1-mask), img*mask, img 

loader = DataLoader(CEDataset('./data/images'), BS, shuffle=True, drop_last=True)

In [ ]:
# Encoder: AlexNet-style → 4000-dim bottleneck
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(ci,co,s=2): return [nn.Conv2d(ci,co,4,s,1), nn.BatchNorm2d(co), nn.LeakyReLU(.2,True)]
        self.net = nn.Sequential(*blk(3,64),*blk(64,128),*blk(128,256),*blk(256,512),*blk(512,512))
        self.fc  = nn.Linear(512*4*4, 4000)
    def forward(self,x): return self.fc(self.net(x).flatten(1))

# Decoder: 4000-dim → 64×64 patch
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(4000,512*4*4), nn.BatchNorm1d(512*4*4), nn.ReLU(True))
        def ubl(ci,co): return [nn.ConvTranspose2d(ci,co,4,2,1), nn.BatchNorm2d(co), nn.ReLU(True)]
        self.net = nn.Sequential(*ubl(512,256),*ubl(256,128),*ubl(128,64),*ubl(64,32),
                                  nn.ConvTranspose2d(32,3,3,1,1), nn.Tanh())
    def forward(self,z): return self.net(self.proj(z).view(-1,512,4,4))

# Discriminator: DCGAN-style, not conditioned on mask
class Disc(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(ci,co,bn=True): return [nn.Conv2d(ci,co,4,2,1),*([ nn.BatchNorm2d(co)] if bn else []),nn.LeakyReLU(.2,True)]
        self.net = nn.Sequential(*blk(3,64,False),*blk(64,128),*blk(128,256),*blk(256,512),
                                  nn.Conv2d(512,1,4,1,0), nn.Sigmoid())
    def forward(self,x): return self.net(x).view(x.size(0))

E, D, Dsc = Encoder().to(device), Decoder().to(device), Disc().to(device)
optG = optim.Adam(list(E.parameters())+list(D.parameters()), LR,   betas=(.5,.999))
optD = optim.Adam(Dsc.parameters(),                           LR/10, betas=(.5,.999))
bce, mse = nn.BCELoss(), nn.MSELoss()
print(f'Params  Enc+Dec: {sum(p.numel() for p in list(E.parameters())+list(D.parameters())):,}  Disc: {sum(p.numel() for p in Dsc.parameters()):,}')

In [ ]:
# Training loop: alternating D and G updates, joint loss L = 0.999*L2 + 0.001*adv
history = []
for ep in range(1, EPOCHS+1):
    E.train(); D.train(); Dsc.train()
    gl = dl = 0
    for masked, gt, full in loader:
        masked, gt, full = masked.to(device), gt.to(device), full.to(device)
        fake = D(E(masked))
        # D step
        optD.zero_grad()
        (bce(Dsc(gt),   torch.ones(BS,device=device)) +
         bce(Dsc(fake.detach()), torch.zeros(BS,device=device))).mul(.5).backward()
        optD.step()
        # G step  (joint loss)
        optG.zero_grad()
        loss = .999*mse(fake,gt) + .001*bce(Dsc(fake), torch.ones(BS,device=device))
        loss.backward(); optG.step()
        gl += loss.item(); dl += 1
    avg = gl/dl; history.append(avg)
    if ep % 10 == 0: print(f'Ep {ep:3d}  loss={avg:.4f}')
print('Done')

In [ ]:
# Visualise results
E.eval(); D.eval()
masked, gt, full = next(iter(loader))
with torch.no_grad(): pred = D(E(masked[:6].to(device))).cpu()
s, e = (IMG-HOLE)//2, (IMG-HOLE)//2+HOLE
composed = masked[:6].clone(); composed[:,:,s:e,s:e] = pred
dn = lambda t: (t*.5+.5).clamp(0,1).permute(1,2,0).numpy()
fig, axes = plt.subplots(3,6,figsize=(15,7))
for i in range(6):
    for j,(img,ttl) in enumerate(zip([masked[i],composed[i],full[i]],['Input','Inpainted','GT'])):
        axes[j,i].imshow(dn(img)); axes[j,i].axis('off')
        if i==0: axes[j,i].set_title(ttl,fontweight='bold')
plt.tight_layout(); plt.savefig('results.png',dpi=120); plt.show()
plt.figure(); plt.plot(history); plt.title('Loss'); plt.xlabel('Epoch'); plt.grid(alpha=.3); plt.show()